In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
# Load Data
df = pd.read_csv("tesla_stock_data.csv")
df["Date"] = pd.to_datetime(df["Date"])
df

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2015-09-21,17.598667,18.104668,17.053333,17.613333,91803000,0.0,0.0
1,2015-09-22,17.268667,17.510000,17.058001,17.396000,54966000,0.0,0.0
2,2015-09-23,17.463333,17.472000,17.172001,17.403999,39012000,0.0,0.0
3,2015-09-24,17.302000,17.563334,17.080667,17.541332,51723000,0.0,0.0
4,2015-09-25,17.774000,17.794001,17.076668,17.127333,56601000,0.0,0.0
...,...,...,...,...,...,...,...,...
2510,2025-09-15,423.130005,425.700012,402.429993,410.040009,163823700,0.0,0.0
2511,2025-09-16,414.500000,423.250000,411.429993,421.619995,104285700,0.0,0.0
2512,2025-09-17,415.750000,428.309998,409.670013,425.859985,106133500,0.0,0.0
2513,2025-09-18,428.869995,432.220001,416.559998,416.850006,90454500,0.0,0.0


In [9]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))

scaled_data = scaler.fit_transform(df['Close'].values.reshape(-1, 1))

In [10]:
look_back = 30
last_sequence = df["Close"].values[-look_back:]

In [12]:
last_sequence_scaled = scaler.transform(last_sequence.reshape(-1, 1))

In [14]:
X_input = np.array([last_sequence_scaled])

In [18]:
# Load my model!
import tensorflow as tf
from tensorflow.keras.models import load_model

model = load_model("MediumModel.keras")

In [20]:
y_pred_scaled = model.predict(X_input)

# Inverse transform back to doller
pred_close = scaler.inverse_transform(y_pred_scaled)[0, 0]

print("Predicted Close for 2025-09-20:", pred_close)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Predicted Close for 2025-09-20: 370.82266


In [53]:
predictions = []
current_sequence = last_sequence_scaled.copy()

for day in range(3):
    X_input = np.array([current_sequence])
    y_pred_scaled = model.predict(X_input, verbose=0)
    pred_close = scaler.inverse_transform(y_pred_scaled)[0, 0]
    predictions.append(pred_close)
    
    pred_scaled_for_update = scaler.transform(np.array([[pred_close]]))
    current_sequence = np.append(current_sequence[1:], pred_scaled_for_update, axis=0)
    
# Print results
dates = pd.date_range(start='2025-09-20', periods=3)
for i, (date, pred) in enumerate(zip(dates, predictions)):
    print(f"Predicted Close for {date.strftime('%Y-%m-%d')}: ${pred: .2f}")

Predicted Close for 2025-09-20: $ 370.82
Predicted Close for 2025-09-21: $ 372.35
Predicted Close for 2025-09-22: $ 371.70
